# Phase 5: LoRA Fine-Tuning Pipeline
**Project:** Python Learning Assistant Using Small Language Model (SLM)  
**Notebook:** `notebooks/phase5_lora_training.ipynb`  
**Base Model:** `Qwen/Qwen2.5-1.5B-Instruct` (Revision: `main`, Apache-2.0)  
**Target Architecture:** Parameter-Efficient Fine-Tuning (PEFT / LoRA, rank=8, alpha=16)  
**Target Runtime:** Google Colab GPU (Tesla T4 or Ampere GPU)  

---

In [ ]:
# Step 1: Hardware Pre-Flight & Colab Environment Audit
import os
import sys
import psutil
import torch

print("=== Colab Runtime Hardware Audit ===")
if not torch.cuda.is_available():
    raise SystemError("FATAL: No CUDA GPU detected in runtime. Select Runtime -> Change runtime type -> T4 GPU.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
cuda_ver = torch.version.cuda
ram_gb = psutil.virtual_memory().total / (1024**3)
avail_ram = psutil.virtual_memory().available / (1024**3)

print(f"GPU Detected       : {gpu_name}")
print(f"Total GPU VRAM     : {vram_gb:.2f} GB")
print(f"CUDA Version       : {cuda_ver}")
print(f"Host System RAM    : {ram_gb:.2f} GB ({avail_ram:.2f} GB available)")

# Determine hardware-optimal precision
# Tesla T4 (SM 7.5 Turing) natively supports FP16; Ampere (SM 8.0+) supports BF16
if torch.cuda.is_bf16_supported():
    TRAIN_DTYPE = torch.bfloat16
    USE_FP16 = False
    USE_BF16 = True
    print("Precision Selected : bfloat16 (Ampere / Hopper GPU)")
else:
    TRAIN_DTYPE = torch.float16
    USE_FP16 = True
    USE_BF16 = False
    print("Precision Selected : float16 (Turing Tesla T4 GPU)")

In [ ]:
# Step 2: Install Compatible Fine-Tuning Dependencies
!pip install -q transformers>=4.40.0 peft>=0.10.0 accelerate>=0.28.0 datasets>=2.18.0 psutil pyyaml

In [ ]:
# Step 3: Dataset Loading & Strict Isolation Check
import json
from pathlib import Path

TRAIN_PATH = Path("data/processed/train.jsonl")
VAL_PATH = Path("data/processed/validation.jsonl")
TEST_PATH = Path("data/processed/test.jsonl")

# Load datasets
with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_data = [json.loads(line) for line in f if line.strip()]

with open(VAL_PATH, "r", encoding="utf-8") as f:
    val_data = [json.loads(line) for line in f if line.strip()]

# Strict Isolation Assertion: Verify test set is completely held out
if TEST_PATH.exists():
    with open(TEST_PATH, "r", encoding="utf-8") as f:
        test_instructions = set(json.loads(line)["instruction"].strip() for line in f if line.strip())
    train_instructions = set(r["instruction"].strip() for r in train_data)
    assert len(train_instructions & test_instructions) == 0, "ERROR: Test set leakage detected!"

print(f"Training Samples   : {len(train_data)}")
print(f"Validation Samples : {len(val_data)}")
print(f"Test Set Isolation : VERIFIED (0 overlapping questions)")

In [ ]:
# Step 4: Tokenizer Initialization & ChatML Formatting with Loss Masking
from transformers import AutoTokenizer
from torch.utils.data import Dataset

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

SYSTEM_PROMPT = """You are an expert, patient, and friendly Python Learning Assistant.
Your mission is to help beginner students master Python programming.
Follow these principles:
1. Explain concepts in clear, simple English without unnecessary jargon.
2. Provide idiomatic, clean, and bug-free Python code examples.
3. Break down code step-by-step or line-by-line when explaining snippets.
4. When diagnosing errors, clearly explain the cause and provide the exact fix.
5. Always encourage the student with beginner-friendly practice questions and hints.
"""

class SFTDataset(Dataset):
    def __init__(self, records, tokenizer, max_len=512):
        self.features = []
        for r in records:
            user_msg = r["instruction"].strip()
            if r.get("input", "").strip():
                user_msg += f"\n\n```python\n{r['input'].strip()}\n```"
            
            prompt_str = f"<|im_start|>system\n{SYSTEM_PROMPT.strip()}<|im_end|>\n<|im_start|>user\n{user_msg}<|im_end|>\n<|im_start|>assistant\n"
            resp_str = f"{r['output'].strip()}<|im_end|>\n"
            
            p_ids = tokenizer.encode(prompt_str, add_special_tokens=False)
            r_ids = tokenizer.encode(resp_str, add_special_tokens=False)
            
            full_ids = p_ids + r_ids
            if len(full_ids) > max_len:
                full_ids = full_ids[:max_len]
            
            # Prompt tokens are masked with -100 so loss is computed ONLY on response tokens
            p_len = min(len(p_ids), len(full_ids))
            labels = [-100] * p_len + full_ids[p_len:]
            mask = [1] * len(full_ids)
            
            self.features.append({
                "input_ids": torch.tensor(full_ids, dtype=torch.long),
                "attention_mask": torch.tensor(mask, dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long)
            })
            
    def __len__(self):
        return len(self.features)
        
    def __getitem__(self, idx):
        return self.features[idx]

train_dataset = SFTDataset(train_data, tokenizer)
val_dataset = SFTDataset(val_data, tokenizer)
print(f"Prepared {len(train_dataset)} train features, {len(val_dataset)} val features with loss masking.")

In [ ]:
# Step 5: Load Base Model and Apply LoRA Adapters
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

print(f"Loading base model {MODEL_ID}...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=TRAIN_DTYPE,
    low_cpu_mem_usage=True
)
base_model.gradient_checkpointing_enable()

# Conservative LoRA Configuration
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

lora_model = get_peft_model(base_model, peft_config)
lora_model.print_trainable_parameters()

In [ ]:
# Step 6: Configure Training Arguments & Launch Trainer
import time
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

OUTPUT_DIR = "artifacts/lora_adapter"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-4,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_ratio=0.05,
    weight_decay=0.01,
    fp16=USE_FP16,
    bf16=USE_BF16,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    seed=42,
    report_to="none"
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True)
)

print("Starting LoRA fine-tuning...")
t_start = time.time()
train_result = trainer.train()
t_total = time.time() - t_start
print(f"Training finished in {t_total:.2f} seconds ({t_total/60:.2f} minutes).")

In [ ]:
# Step 7: Measure Final Validation Loss & Evaluate Overfitting
import math

eval_metrics = trainer.evaluate()
val_loss = eval_metrics.get("eval_loss", 0.0)
val_ppl = math.exp(min(val_loss, 20.0))

print(f"Final Validation Loss       : {val_loss:.4f}")
print(f"Final Validation Perplexity : {val_ppl:.2f}")

# Step 8: Save LoRA Adapter Artifacts
lora_model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved trained LoRA adapter to {OUTPUT_DIR}/")